# EPIC Clarity Device Exposure Hydration

This notebook hydrates the OMOP DEVICE_EXPOSURE table from EPIC Clarity OR implant data.

## Source Table
- `_exponent._bronze_epic_clarity_*.dbo_OR_IMP`

In [0]:
source = 'epic_clarity'

In [0]:
silver_device_exposure_df = spark.sql(f'''
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'OR_IMP', 'IMPLANT_ID', oi.IMPLANT_ID) AS device_source_value,
    oi.IMPLANT_NAME AS device_name,
    oi.STATUS_C_NAME AS device_type_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.or_imp oi
WHERE oi.IMPLANT_ID IS NOT NULL
''')

display(silver_device_exposure_df)
silver_device_exposure_df.createOrReplaceTempView("silver_device_exposure")

In [0]:
%sql
MERGE INTO _exponent.omop_silver.device_exposure AS target
USING silver_device_exposure AS source
ON target.device_exposure_source_value = source.device_exposure_source_value

WHEN MATCHED AND NOT (
    target.device_source_value <=> source.device_source_value
)
THEN UPDATE SET
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    device_exposure_source_value,
    updated_tsp
)
VALUES (
    source.device_exposure_source_value,
    source.updated_tsp
)

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_device_exposure (
    source_system,
    device_exposure_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.device_exposure_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.updated_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 'epic_clarity' AS source_system, device_exposure_source_value, updated_tsp
    FROM _exponent.omop_silver.device_exposure
    WHERE device_exposure_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_device_exposure x
    ON s.device_exposure_source_value = x.device_exposure_source_value
    AND x.source_system = 'epic_clarity'

In [0]:
gold_device_exposure_df = spark.sql("""
SELECT
    m.device_exposure_id,
    s.updated_tsp
FROM _exponent.omop_silver.device_exposure s
INNER JOIN _exponent.omop_mapping.source_to_device_exposure m
    ON s.device_exposure_source_value = m.device_exposure_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE
""")

display(gold_device_exposure_df)
gold_device_exposure_df.createOrReplaceTempView("gold_device_exposure")

In [0]:
%sql
-- MERGE INTO _exponent.omop.device_exposure AS target
MERGE INTO _exponent.omop_epic.device_exposure AS target
USING gold_device_exposure AS source
ON target.device_exposure_id = source.device_exposure_id

WHEN NOT MATCHED THEN INSERT (
    device_exposure_id
)
VALUES (
    source.device_exposure_id
)